# What Data Do We Have? A Guided Tour

This notebook walks through every dataset in the toolkit. Run each cell and look at the data with your domain expertise. I've flagged questions where **YOUR knowledge matters more than any model**.

Cells marked with **SAM'S EXPERTISE NEEDED** are places where the data raises a question only you can answer -- your LC-MS experience, your knowledge of the degradation mechanisms, your intuition about what matters.

**Run from the `bioai-toolkit/` directory** (the notebook assumes paths relative to the parent of `notebooks/`).

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Ensure we can import from the toolkit
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
# If running from bioai-toolkit already, no change needed
if not os.path.exists("data"):
    os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))

sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly not installed -- falling back to matplotlib for all plots")

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

print(f"Working directory: {os.getcwd()}")
print(f"Plotly available: {HAS_PLOTLY}")
print(f"Seaborn available: {HAS_SEABORN}")

---
## 1. Rozans 618 Enriched -- Your Peptide Library

This is the core dataset: 618 peptides from your three papers, enriched with computed physicochemical and exopeptidase susceptibility features.

In [ ]:
rozans = pd.read_csv("data/rozans-618-enriched.csv")

print(f"Shape: {rozans.shape[0]} peptides x {rozans.shape[1]} features")
print(f"\nPapers:")
for paper, count in rozans["paper"].value_counts().items():
    print(f"  {paper}: {count} peptides")

print(f"\nUnique sequences: {rozans['clean_sequence'].nunique()}")
print(f"Unique scaffolds: {rozans['scaffold'].nunique()}")

print(f"\nN-terminal modifications:")
for mod, count in rozans["n_terminal"].value_counts().items():
    print(f"  {mod}: {count}")

print(f"\nC-terminal modifications:")
for mod, count in rozans["c_terminal"].value_counts().items():
    print(f"  {mod}: {count}")

print(f"\nKey computed features (first 5 rows):")
rozans[["sequence", "paper", "aminopeptidase_susceptibility", 
        "carboxypeptidase_susceptibility", "total_exopeptidase_susceptibility",
        "mmp_cleavage_score"]].head()

### SAM'S EXPERTISE NEEDED

Which of these 618 peptides had the **most surprising degradation behavior** in your experiments? That's where the model should focus -- the peptides where intuition fails are exactly where ML adds value.

Write your notes here:
- Peptide(s): 
- What was surprising: 
- Hypothesis for why: 

---
## 2. MEROPS Exopeptidase Cleavages

15,883 cleavage events from the MEROPS database, filtered to exopeptidase families relevant to your work.

In [ ]:
merops = pd.read_csv("data/processed/merops_exopeptidase_cleavages.csv")

print(f"Shape: {merops.shape[0]} cleavage events x {merops.shape[1]} columns")
print(f"\nCleavage types:")
for ct, count in merops["cleavage_type"].value_counts().items():
    print(f"  {ct}: {count}")

print(f"\nTop 10 protease families by cleavage count:")
family_counts = merops["protease_family"].value_counts()
for fam, count in family_counts.head(10).items():
    # Get a representative protease name for this family
    example = merops[merops["protease_family"] == fam]["protease_name"].iloc[0]
    print(f"  {fam}: {count:,} cleavages (e.g., {example})")

print(f"\nTotal unique proteases: {merops['protease_name'].nunique()}")
print(f"Total unique substrates: {merops['substrate_name'].nunique()}")

In [ ]:
# P1 amino acid frequency heatmap by protease family
# P1 is the residue immediately N-terminal to the cleavage site --
# this is what the exopeptidase "sees" as it chews from the terminus.

from exopred.data_pipeline import _three_to_one

STANDARD_AAS = list("ACDEFGHIKLMNPQRSTVWY")

# Convert P1 3-letter codes to 1-letter
merops["P1_1"] = merops["P1"].apply(
    lambda x: _three_to_one(str(x).strip()) if pd.notna(x) else None
)

# Get top 8 families for readability
top_families = merops["protease_family"].value_counts().head(8).index.tolist()
sub = merops[merops["protease_family"].isin(top_families) & merops["P1_1"].isin(STANDARD_AAS)]

# Build frequency matrix
freq = pd.crosstab(sub["P1_1"], sub["protease_family"], normalize="columns")
freq = freq.reindex(index=STANDARD_AAS, columns=top_families).fillna(0)

fig, ax = plt.subplots(figsize=(10, 8))
if HAS_SEABORN:
    sns.heatmap(freq, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, 
                linewidths=0.5, cbar_kws={"label": "Frequency"})
else:
    im = ax.imshow(freq.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(top_families)))
    ax.set_xticklabels(top_families, rotation=45)
    ax.set_yticks(range(len(STANDARD_AAS)))
    ax.set_yticklabels(STANDARD_AAS)
    plt.colorbar(im, ax=ax, label="Frequency")

ax.set_title("P1 Amino Acid Frequency by Exopeptidase Family\n(what residue is at the cleavage site?)")
ax.set_ylabel("P1 Amino Acid")
ax.set_xlabel("Protease Family")
plt.tight_layout()
plt.show()

### SAM'S EXPERTISE NEEDED

The heatmap above shows what MEROPS says about P1 preferences per exopeptidase family. **Do these match what you saw in your LC-MS data?** 

Key questions:
- C01 (cathepsins) dominate the dataset. Are cathepsins even relevant in your cell culture systems (hMSC, hUVEC, macrophage, THP-1)?
- M24 (methionine aminopeptidases) are second. These only clip Met -- is that relevant for your RGEFV scaffolds?
- Where do the MEROPS frequencies **disagree** with your experimental observations?

Notes:
- Agreement: 
- Disagreement: 
- Proteases missing from MEROPS that matter in your system: 

---
## 3. PEPlife2 -- Peptide Half-Life Database

4,500 peptide half-life measurements from PEPlife2. Heterogeneous conditions -- many proteases, species, assays. **Handle with care.**

In [ ]:
peplife = pd.read_csv("data/processed/peplife2_combined.csv")

print(f"Shape: {peplife.shape[0]} entries")
print(f"Unique peptides: {peplife['seq'].nunique()}")
print(f"\nProtease/system types (top 15):")
for prot, count in peplife["protease"].value_counts().head(15).items():
    print(f"  {prot}: {count}")

print(f"\nHalf-life unit mess (this is why it's tricky):")
for unit, count in peplife["units_half"].value_counts().items():
    print(f"  '{unit}': {count}")

# Normalize to minutes for the histogram
unit_map = {
    "minutes": 1.0, "minute": 1.0, "Minutes": 1.0,
    "hours": 60.0, "hour": 60.0, "Hours": 60.0,
    "seconds": 1/60, "Seconds": 1/60,
    "days": 1440.0, "day": 1440.0, "Days": 1440.0,
}
pep_valid = peplife[peplife["units_half"].isin(unit_map.keys()) & peplife["half_life"].notna()].copy()
pep_valid["half_life_min"] = pep_valid["half_life"] * pep_valid["units_half"].map(unit_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale half-life distribution
axes[0].hist(np.log10(pep_valid["half_life_min"].clip(lower=0.01)), bins=50, 
             color="steelblue", edgecolor="white")
axes[0].set_xlabel("log10(half-life in minutes)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Half-Life Distribution (n={len(pep_valid)})")
axes[0].axvline(np.log10(2880), color="red", linestyle="--", label="48 hours")
axes[0].legend()

# In vivo vs in vitro
vivo_counts = peplife["vivo_vitro"].value_counts()
axes[1].bar(vivo_counts.index, vivo_counts.values, color=["#2196F3", "#FF9800", "#4CAF50"][:len(vivo_counts)])
axes[1].set_title("In Vivo vs In Vitro")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\nWARNING: {peplife.shape[0] - pep_valid.shape[0]} entries have non-standard units or missing half-lives")

### SAM'S EXPERTISE NEEDED

PEPlife2 has half-lives measured under wildly different conditions. The model currently uses it only for **validation** (Spearman correlation), not training.

**Are any of these proteases/conditions close enough to your cell culture system to use as training data?**

- "Murine crude intestinal proteases" -- clearly different from hMSC conditioned media
- "Human plasma protease" -- closer to hUVEC?
- "Trypsin" -- is trypsin a good proxy for anything in your system?

Which PEPlife2 subsets could actually augment your training data?
- Useful subset: 
- Why: 

---
## 4. Turk 2015 -- 18,583 MMP Substrate Peptides

Z-scores for cleavage of 18,583 peptide substrates across 18 MMP isoforms. This is the largest systematic MMP specificity dataset available. Directly relevant to your Paper 3 (MMP-14 crosslinkers).

In [ ]:
turk = pd.read_excel("data/turk2015/mmc2-table-S1.xlsx")
turk = turk.rename(columns={"Unnamed: 0": "sequence"})

mmp_cols = [c for c in turk.columns if c.startswith("MMP")]
print(f"Shape: {turk.shape[0]:,} peptides x {len(mmp_cols)} MMP isoforms")
print(f"MMP isoforms: {mmp_cols}")
print(f"Sequence length: {turk['sequence'].str.len().describe()}")

# Z-score distribution for MMP-14 specifically (Paper 3 target)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(turk["MMP14"], bins=80, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="gray", linestyle="--")
axes[0].axvline(2, color="red", linestyle="--", label="Z>2 (strong substrate)")
axes[0].set_xlabel("MMP-14 Z-score")
axes[0].set_ylabel("Count")
axes[0].set_title(f"MMP-14 Z-Score Distribution (n={len(turk):,})")
axes[0].legend()
print(f"\nMMP-14 strong substrates (Z>2): {(turk['MMP14'] > 2).sum()} peptides")
print(f"MMP-14 weak/non-substrates (Z<0): {(turk['MMP14'] < 0).sum()} peptides")

# P1 position amino acid preferences for MMP-14 (position 5 of 10-mer)
# Turk sequences are 10-mers; cleavage between positions 5-6 (P1-P1')
turk["P1_aa"] = turk["sequence"].str[4]  # 0-indexed position 4 = P1
p1_zscore = turk.groupby("P1_aa")["MMP14"].mean().sort_values(ascending=False)

colors = ["#d32f2f" if z > 0.5 else "#1976d2" if z < -0.5 else "#757575" for z in p1_zscore.values]
axes[1].bar(p1_zscore.index, p1_zscore.values, color=colors)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_xlabel("P1 Amino Acid")
axes[1].set_ylabel("Mean MMP-14 Z-score")
axes[1].set_title("MMP-14 P1 Preferences (Turk 2015)")

plt.tight_layout()
plt.show()

print(f"\nTop 5 P1 residues for MMP-14:")
for aa, z in p1_zscore.head(5).items():
    print(f"  {aa}: mean Z = {z:.3f}")

### SAM'S EXPERTISE NEEDED

Your Paper 3 is about MMP-14 crosslinkers. **Does the Turk data match your experimental ranking?**

- Do the top P1 residues above (highest Z-scores) align with which crosslinker sequences worked best in your split-and-pool screen?
- The Turk assay uses purified MMPs on short peptides. Your system has whole cells producing MMPs in a matrix. How much do you expect the rankings to transfer?

Notes:
- Agreement with your data: 
- Differences: 
- Would you trust Turk Z-scores to pre-filter a new crosslinker library? (Y/N and why): 

---
## 5. DPP-IV Benchmark

1,330 peptides labeled as DPP-IV inhibitors or non-inhibitors. This is the **commercial angle** dataset -- DPP-IV inhibition is the biggest market in peptide stability (GLP-1/diabetes drugs).

In [ ]:
dppiv = pd.read_csv("data/processed/dppiv_benchmark.csv")

print(f"Shape: {dppiv.shape[0]} peptides")
print(f"\nClass balance:")
for label, count in dppiv["label"].value_counts().items():
    label_name = "Inhibitor" if label == 1 else "Non-inhibitor"
    print(f"  {label_name} ({label}): {count} ({count/len(dppiv)*100:.1f}%)")

print(f"\nTrain/test split:")
for split, count in dppiv["split"].value_counts().items():
    print(f"  {split}: {count}")

# Sequence length distribution
dppiv["length"] = dppiv["sequence"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(1, "#4CAF50", "Inhibitor"), (0, "#f44336", "Non-inhibitor")]:
    sub = dppiv[dppiv["label"] == label]
    axes[0].hist(sub["length"], bins=range(2, 30), alpha=0.6, color=color, label=name, edgecolor="white")
axes[0].set_xlabel("Sequence Length")
axes[0].set_ylabel("Count")
axes[0].set_title("DPP-IV Peptide Length Distribution")
axes[0].legend()

# N-terminal dipeptide (DPP-IV cleaves after position 2)
dppiv["dipeptide"] = dppiv["sequence"].str[:2]
top_dipeptides = dppiv[dppiv["label"] == 1]["dipeptide"].value_counts().head(15)
axes[1].barh(top_dipeptides.index[::-1], top_dipeptides.values[::-1], color="#4CAF50")
axes[1].set_xlabel("Count")
axes[1].set_title("Top N-terminal Dipeptides in DPP-IV Inhibitors")

plt.tight_layout()
plt.show()

### SAM'S EXPERTISE NEEDED

**Strategic question: If you were building an ExoPred product, would you start with exopeptidases (your scientific moat) or DPP-IV (biggest market)?**

- Exopeptidase path: unique data, less competition, smaller initial market
- DPP-IV path: huge pharma market (GLP-1 drugs), more competition, bigger revenue potential
- Hybrid: use exopeptidase model as a feature in DPP-IV prediction?

Your thinking: 

---
## 6. Bottger External Validation -- Antimicrobial Peptides

55 measurements across 22 peptides from Bottger et al. (2017). These are antimicrobial peptides tested in human plasma, blood, and serum. The **only truly independent validation set** we have.

In [ ]:
bottger = pd.read_csv("data/external_validation/extracted_data.csv")
cleavage = pd.read_csv("data/external_validation/cleavage_sites.csv")

print(f"Stability data: {bottger.shape[0]} measurements across {bottger['peptide_name'].nunique()} peptides")
print(f"Cleavage site data: {cleavage.shape[0]} identified metabolites")

print(f"\nMatrices tested:")
for matrix, count in bottger["matrix"].value_counts().items():
    print(f"  {matrix}: {count}")

print(f"\nPeptides and their stability (median % intact across conditions):")
pep_summary = bottger.groupby("peptide_name").agg(
    n_conditions=("matrix", "count"),
    median_pct_intact=("pct_intact", "median"),
    min_pct_intact=("pct_intact", "min"),
    max_pct_intact=("pct_intact", "max"),
    has_halflife=("half_life", lambda x: x.notna().sum()),
).sort_values("median_pct_intact", ascending=False)

print(pep_summary.to_string())

print(f"\n\nCleavage types observed:")
for ct, count in cleavage["cleavage_type"].value_counts().items():
    print(f"  {ct}: {count}")

# Key: these peptides have KNOWN cleavage sites -- ground truth for mechanism
print(f"\nPeptides with identified cleavage sites: {cleavage['parent_peptide'].unique()}")

### SAM'S EXPERTISE NEEDED

These are antimicrobial peptides -- structurally very different from your adhesion peptides. But they have the **best-characterized degradation mechanisms** (identified cleavage sites, multiple matrices).

**Any structural similarities to your adhesion peptides that might transfer?**
- Pro-rich regions? (several Bottger peptides are Pro-rich)
- Terminal modifications? (some have pyroglutamate, some are amidated)
- Length range overlap?

Notes:
- Transferable features: 
- Key differences that limit transfer: 

---
## 7. Summary: All Datasets at a Glance

In [ ]:
summary = pd.DataFrame({
    "Dataset": [
        "Rozans 618 (enriched)",
        "MEROPS exopeptidase",
        "PEPlife2",
        "Turk 2015 MMP",
        "DPP-IV benchmark",
        "Bottger validation",
        "Bottger cleavage sites",
        "ESM-2 embeddings",
    ],
    "Records": [618, 15883, 4500, 18583, 1330, 55, 23, 19],
    "Role in Pipeline": [
        "Core training data (Paper 1 = labels, Paper 3 = MMP)",
        "Feature source (cleavage frequencies by family)",
        "Independent validation (Spearman correlation)",
        "MMP-14 specificity features (v4 model)",
        "Commercial benchmark (DPP-IV product)",
        "External validation (zero correlation currently)",
        "Mechanistic ground truth (where peptides actually cleave)",
        "Protein language model features (19 Paper 1 sequences)",
    ],
    "Quality/Concern": [
        "Labels are COMPUTED from calibration model, not measured",
        "Dominated by C01 (cathepsins) -- may not match your proteases",
        "Heterogeneous conditions, messy units",
        "Purified enzyme assay -- may not transfer to cell culture",
        "Binary labels only, no degradation kinetics",
        "Only 22 peptides, antimicrobial (different domain)",
        "Small but gold-standard mechanistic data",
        "Only 19 sequences -- need to expand if using ESM features",
    ],
})

# Display as a formatted table
print(summary.to_string(index=False))
print(f"\n{'='*80}")
print(f"TOTAL: ~41K data points across 8 datasets")
print(f"BUT: Only 618 peptides are actually YOURS (and labels are computed, not measured)")
print(f"The 80K dataset will change everything.")